In [ ]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.1 MB/s eta 0:00:00


# **Built-in Tool - DuckDuckGo Search**
---
Say u asked llm a current affair question where its has got no parameteric knowledge due to its knowledge cutoff date so if we give llm access to a tool which asks a search engine or does web search....:

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool = DuckDuckGoSearchRun()  #Runnable
result = search_tool.invoke("argentina messi final match news")
print(result)

Argentina bids Messi farewell in final national team match, ending an era Argentina's Lionel Messi celebrates after scoring his side's third goal from the penalty spot during his final match for the national team, a soccer friendly against Benin, in Buenos Aires, Argentina, Tuesday, Oct.6, 2026. (AP/Photo/Gustavo Garello) License this photo BUENOS AIRES, Argentina -- Argentina is confronting a once-unimaginable reality: a national team without soccer great Lionel Messi. On Tuesday, 39-year-old Messi bid farewell to international soccer after 21 years, 126 goals, two Copa América titles and a World Cup championship, scoring a penalty ... Lionel Messi signed off from international football with a goal on Tuesday night in a 3-0 friendly win against Benin in Buenos Aires. Messi's 208th and final game for Argentina ended with his ... Lionel Messi marked his 208th and final game for Argentina with his 126th goal as his country bid an emotional farewell to the global icon. Gustavo Garello Arg

# **Built-in Tool - Shell Tool**

In [ ]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
sample_data



/usr/local/lib/python3.13/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


In [ ]:
res = shell_tool.invoke('whoami')
print(res) #where this code is running maybe a google server where its name is root

Executing command:
 whoami
root



# **Custom Tool**

In [ ]:
from langchain_core.tools import tool

In [ ]:
# Step 1 - create a function

#say ur llm doesn't know how to do math so let's create an external funciton first
def multiply(a, b):
    """Multiply two numbers""" #(not mandatory but highly recomended cz:)let these doc strings be mentioned so that it makes llm understand what this function or the tool does
    return a*b

In [ ]:
  # Step 2 - add type hints (not mandatory but highly recomended cz:)
# so that llm will get to which type of data to take as input and what type to give as output
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [ ]:
# Step 3 - add tool decorator
# so that it makes the function a special function which lets llm to communicate with it
@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""  #doc string is mandatory after tool decorator added. run without this if u want to check the message of error.
    return a*b

In [ ]:
#after 3 steps, tool has been created but how to use it:
result = multiply.invoke({"a":3, "b":5}) #function name is the tool name and it is a runnable uk!
print(result)

15


In [ ]:
#it seems like a normal function but it isn't and u will learn that in tool calling.
#tool have special properties:
print(multiply.name) #tool's name
print(multiply.description) #doc string what u provided since description was not provided
print(multiply.args)  #arguments

#and u will find these 3 attributes with any tool:

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [ ]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


In [ ]:
#when llm connects to tool we will not send the logic to llm rather a schema of tool is sent to llm:
print(multiply.args_schema.model_json_schema())
#copy the output and paste it in websearch:"json formatter and validator" and process it to see the schema with proper indentation

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


**## Method 2 - Using StructuredTool**

In [ ]:
from langchain_community.tools import StructuredTool
from pydantic import BaseModel, Field

In [ ]:
#defining the schema
class MultiplyInput(BaseModel):
  a:int = Field(required="true", description="The first number to add")
  b:int = Field(required="true", description="The second number to add")

/tmp/ipykernel_956/767862655.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a:int = Field(required="true", description="The first number to add")
/tmp/ipykernel_956/767862655.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b:int = Field(required="true", description="The second number to add")


In [ ]:
def multiply_func(a,b): #u can add type hinting here too but not required since we gave everything we want in schema
  return a*b

In [ ]:
#here u r creating a tool (obj)
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput #here u pass the arg schema or enforce it here
)

In [ ]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


**Method 3 - Using BaseTool Class**

In [ ]:
from langchain.tools import BaseTool
from typing import Type #Type is used when you want to say: "This variable should contain a class, not an object."

In [ ]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_956/908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_956/908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [ ]:
#here we r creating a tool which inherits from BaseTool
class MultiplyTool(BaseTool): 
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput 
    
    def _run(self, a: int, b: int) -> int: 
        return a * b

    #simple understanding is that _run is an abstract method of abstract class BaseTool hence the required defining of _run method here and
    #name:str, args_schema: Type[BaseModel] - mentioning type annotations was mandatory cz it has something to do with BaseTool class too -
    #LangChain's BaseTool is built using Pydantic's model system. Pydantic uses type annotations to define the fields of a model.

In [ ]:
multiply_tool = MultiplyTool()

In [ ]:
result = multiply_tool.invoke({'a':3, 'b':3}) #validation by args_schema and then basetool expects it's sub classes to implement _run()

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


#**ToolKit**

In [ ]:
#custom tools:
@tool
def add(a: int, b:int)->int:
  """add two numbers"""
  return a+b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [ ]:
#u create a class for whatever toolkit u want to create
class MathToolkit:
    def get_tools(self):
        return [add, multiply] #return the names of the functions which u want those to be a part of the toolkit

In [ ]:
#create an obj of toolkit and u get the tools from get_tools method of MathToolkit and then u can access,run the tools by running a loop
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

add => add two numbers
multiply => Multiply two numbers
